# 03 - Feature Selection

このノートブックは、最も予測力のある特徴を選択します。

## Contents
1. データ読み込み
2. 単変量検定（相関、相互情報量）
3. モデルベースの特徴重要度
4. 特徴選択結果の統合
5. 最終特徴セットの保存

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Paths
TRAIN_PATH = 'data/processed/train_engineered.csv'
TEST_PATH = 'data/processed/test_engineered.csv'
PROCESSED_DIR = Path('data/processed')
ANALYSIS_DIR = Path('data/analysis')

print('Imports successful!')

## 1. データ読み込み

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

target_col = 'Irrigation_Need'
y_train = train_df[target_col].values
X_train = train_df.drop([target_col], axis=1)
X_test = test_df.copy()

print(f'Train shape: {X_train.shape}')
print(f'Target distribution: {np.bincount(y_train)}')
print(f'Features: {X_train.shape[1]}')

## 2. 単変量検定：相互情報量

In [ ]:
# Calculate mutual information
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({
    'Feature': X_train.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print('Top 20 Features by Mutual Information:')
print(mi_df.head(20))

# Visualization
plt.figure(figsize=(10, 8))
plt.barh(range(20), mi_df.head(20)['MI_Score'].values)
plt.yticks(range(20), mi_df.head(20)['Feature'].values)
plt.xlabel('Mutual Information Score')
plt.title('Top 20 Features by Mutual Information')
plt.tight_layout()
plt.savefig('mi_scores.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. モデルベース特徴重要度（Random Forest）

In [ ]:
# Train Random Forest for feature importance
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rf_model.fit(X_train, y_train)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('\nTop 20 Features by Random Forest Importance:')
print(importance_df.head(20))

# Visualization
plt.figure(figsize=(10, 8))
plt.barh(range(20), importance_df.head(20)['Importance'].values)
plt.yticks(range(20), importance_df.head(20)['Feature'].values)
plt.xlabel('Importance Score')
plt.title('Top 20 Features by Random Forest')
plt.tight_layout()
plt.savefig('rf_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. 特徴選択戦略

In [ ]:
# Combine scores
combined_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'MI_Rank': mi_df.set_index('Feature').reindex(X_train.columns).index.to_series().rank(ascending=False),
    'RF_Rank': importance_df.set_index('Feature').reindex(X_train.columns).index.to_series().rank(ascending=False)
})

# Normalize ranks
combined_scores['MI_Score'] = mi_df.set_index('Feature').reindex(X_train.columns)['MI_Score'].values
combined_scores['RF_Score'] = importance_df.set_index('Feature').reindex(X_train.columns)['Importance'].values

# Normalize to 0-1
combined_scores['MI_Normalized'] = (combined_scores['MI_Score'] - combined_scores['MI_Score'].min()) / \
                                    (combined_scores['MI_Score'].max() - combined_scores['MI_Score'].min() + 1e-8)
combined_scores['RF_Normalized'] = (combined_scores['RF_Score'] - combined_scores['RF_Score'].min()) / \
                                    (combined_scores['RF_Score'].max() - combined_scores['RF_Score'].min() + 1e-8)

# Combined score
combined_scores['Combined_Score'] = (combined_scores['MI_Normalized'] + combined_scores['RF_Normalized']) / 2
combined_scores = combined_scores.sort_values('Combined_Score', ascending=False)

print('Top 20 Features by Combined Score:')
print(combined_scores.head(20)[['Feature', 'Combined_Score']])

# Select top features
# Strategy: Keep features with combined score > median or top N features
threshold = combined_scores['Combined_Score'].quantile(0.5)  # Top 50%
selected_features = combined_scores[combined_scores['Combined_Score'] >= threshold]['Feature'].tolist()

print(f'\nSelected {len(selected_features)} features (top 50% by combined score)')

## 5. 最終特徴セットの保存

In [ ]:
# Create final datasets with selected features
train_selected = X_train[selected_features].copy()
train_selected[target_col] = y_train

test_selected = X_test[selected_features].copy()

# Save
train_selected.to_csv(PROCESSED_DIR / 'train_engineered_selected.csv', index=False)
test_selected.to_csv(PROCESSED_DIR / 'test_engineered_selected.csv', index=False)

print(f'✓ Train selected: {train_selected.shape}')
print(f'  Saved to: data/processed/train_engineered_selected.csv')
print(f'\n✓ Test selected: {test_selected.shape}')
print(f'  Saved to: data/processed/test_engineered_selected.csv')

# Save feature importance
feature_importance_dict = combined_scores[['Feature', 'Combined_Score']].set_index('Feature')['Combined_Score'].to_dict()
with open(ANALYSIS_DIR / 'feature_importance.json', 'w') as f:
    json.dump(feature_importance_dict, f, indent=2)

print(f'\n✓ Feature importance saved to: data/analysis/feature_importance.json')
print('\n=== Feature Selection Complete ===')